# ဒေသဆိုင်ရာ ၇: စကားပြော အက်ပ်လီကေးရှင်းများ တည်ဆောက်ခြင်း
## Azure OpenAI API အမြန်စတင်ခြင်း


## Overview
This notebook is adapted from the [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) that includes notebooks that also access the [OpenAI](oai-assignment.ipynb) service.

The Python OpenAI API works with Azure OpenAI as well, with a few modifications. Learn more about the differences here: [How to switch between OpenAI and Azure OpenAI endpoints with Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

For more quickstart examples please refer to the official [Azure OpenAI Quickstart Documentation](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst)


## အကြောင်းအရာ စာရင်း  

[အကြောင်းအရာအနှိုင်းအယှက်](#overview)  
[Azure OpenAI Service ဖြင့် စတင်အသုံးပြုခြင်း](#getting-started-with-azure-openai-service)  
[သင်၏ ပထမဆုံး prompt တည်ဆောက်ခြင်း](#build-your-first-prompt)  

[အသုံးချမှုများ](#use-cases)    
[၁။ စာသားကို အကျဉ်းချုပ်တင်ပြခြင်း](#summarize-text)  
[၂။ စာသားကို အမျိုးအစားခွဲခြားခြင်း](#classify-text)  
[၃။ ပစ္စည်းအသစ်ခေါင်းစဉ်များ ဖန်တီးခြင်း](#generate-new-product-names)  
[၄။ အမျိုးအစားခွဲစက်တစ်ခု အဆင့်မြှင့်လုပ်ခြင်း](#fine-tune-a-classifier)  
[၅။ အင်ဘက်ဒ်များ](#embeddings)

[ရင်းမြစ်များ](#references)


### Azure OpenAI ဝန်ဆောင်မှုနှင့်အတူ စတင်ခြင်း

ဖောက်သည်အသစ်များသည် Azure OpenAI ဝန်ဆောင်မှုအတွက် [လက်မှတ်တောင်းခံရန်](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst) လိုအပ်ပါသည်။  
ခွင့်ပြုချက်စစ်ဆေးမပြီးဆုံးသည့်နောက်၊ ဖောက်သည်များသည် Azure ပေါ်တယ်သို့လော့ဂ်အင်ဝင်၍ Azure OpenAI ဝန်ဆောင်မှုရင်းမြစ်တစ်ခုကို ဖန်တီးနိုင်ပြီး studio မှတဆင့်ပုံစံများနှင့် စမ်းသပ်နိုင်ပါသည်  

[လျင်မြန်စွာစတင်ရန်အတွက် ကောင်းမွန်သောအရင်းအမြစ်](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### သင့်ပထမဆုံး prompt ကို တည်ဆောက်ပါ  
ဒီတိုတောင်းတဲ့လေ့ကျင့်မှုက လွယ်ကူတဲ့ "အကျဉ်းချုပ်ပေးခြင်း" အလုပ်အတွက် OpenAI မော်ဒယ်ကို prompt များ တင်သွင်းပုံအကြောင်း အခြေခံမိတ်ဆက်ပေးပါမည်။


**လမ်းဆုံများ**:  
1. သင့် Python ပတ်ဝန်းကျင်မှာ OpenAI 라이ဘ्रेရီကို ထည့်သွင်းပါ  
2. စံပေါ် ကူညီမည့် 라이ဘ뢰ရီများကို လုပ်ဆောင်ပြီး သင့်ဖန်တီးထားသော OpenAI ဝန်ဆောင်မှု အတွက် OpenAI လုံခြုံရေးအချက်အလက်များကို သတ်မှတ်ပါ  
3. သင့်အလုပ်အတွက် မော်ဒယ်ကို ရွေးပါ  
4. မော်ဒယ်အတွက် လွယ်ကူသော prompt ကို တည်ဆောက်ပါ  
5. မော်ဒယ် API ကို သင့်တောင်းဆိုမှုကို တင်ပြပါ!


### 1. OpenAI ကို အပ်လိတ်လုပ်ရန်


  > [!NOTE] Codespaces သို့မဟုတ် Devcontainer ထဲတွင် ဤ notebook ကို လည်ပတ်ပါက ဤအဆင့်သည် မလိုအပ်ပါ


In [ ]:
%pip install openai python-dotenv

### 2. အကူအညီစာကြည့်တိုက်များကို import လုပ်ပြီး အတည်ပြုချက်များကို instantiate ပြုလုပ်ပါ


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. သင့်တော်သော မော်ဒယ် ရှာဖွေရန်  
GPT-4o နှင့် GPT-4o mini ကဲ့သို့သော မော်ဒယ်များသည် သဘာဝဘာသာစကားကို နားလည်နိုင်ပြီး ဖန်တီးထုတ်လုပ်နိုင်သည်။ Microsoft Foundry အတွင်းရှိ Azure OpenAI သည် မတူညီသော လုပ်ဆောင်ချက်များအတွက် သင့်တော်သော အင်အားနှင့် အမြန်နှုန်းအဆင့်များ ဖြင့် မော်ဒယ်စွမ်းဆောင်ရည် များစွာကို ပေးအပ်သည်။  

[Azure OpenAI in Microsoft Foundry models](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## 4. အကြံပြုချက် ဒီဇိုင်း  

"အကြီးစားဘာသာစကားမော်ဒယ်များ၏ စွမ်းဆောင်ရည်အံ့သြဖွယ်မှာ စာသားများစွာပေါ်တွင် မျှော်မှန်းချက်အမှားကို ကျဆင်းအောင်လေ့ကျင့်ခြင်းဖြင့် မော်ဒယ်များသည် ဤမျှော်မှန်းချက်များအတွက် အသုံးဝင်သော အယူအဆများကို သင်ယူသွားကြသည်။ ဥပမာ၏ အနေနှင့်၊ ၎င်းတို့သည် အောက်ပါအယူအဆများကို သင်ယူသွားကြသည်"(1)-

* စာလုံးပေါင်းရေးနည်း
* စာပေသတ္တိလမ်းညွှန်ချက်
* ပြန်ဆိုနည်း
* မေးခွန်းများကို ဖြေဆိုနည်း
* စကားပြောဆိုနည်း
* ဘာသာစကားများစွာဖြင့် ရေးသားနည်း
* ကုဒ်ရေးနည်း
* အစရှိသည်များ

#### အကြီးစားဘာသာစကားမော်ဒယ်ကို ထိန်းချုပ်နည်း  
"အကြီးစားဘာသာစကားမော်ဒယ်တွင် ပါဝင်သော input များအားလုံးကြားမှ အကျိုးသက်ရောက်မှုအရှိဆုံးက ဇာတ်ကိုဇာတ်လမ်း prompt(1) ဖြစ်သည်။

အကြီးစားဘာသာစကားမော်ဒယ်များကို အောက်ပါနည်းလမ်းများဖြင့် output ထုတ်ပေးရန် အကြံပြုနိုင်သည်-

- ညွှန်ကြားချက် ။ မော်ဒယ်အား သင်လိုချင်သည့် အကြောင်းပြချက်ပြောပြသည်
- ပြည့်စုံမှု ။ မော်ဒယ်ကို သင့်လိုချင်သော အစပိုင်းကို ပြည့်စုံအောင် ဖိအားပေးသည်
- ဖော်ပြချက် ။ မော်ဒယ်အား သင်လိုချင်သည့် အချက်ကို ပြသသည်၊ အောက်ပါနှစ်ခုမှတစ်ခုဖြစ်နိုင်သည်-
- ရှေ့တည်နေသော ပရောမ့်တွင် နမူနာအနည်းငယ်
- ကျွမ်းကျင်သင်ကြားရေး dataset ထဲတွင် နမူနာရာနှင့်ထက်များစွာ



#### ပရောမ့်များ ဖန်တီးရာတွင် အခြေခံလမ်းညွှန်ချက်သုံးချက်ရှိသည်-

**ပြသပြီး ပြောပြပါ** ။ ညွှန်ကြားချက်များ၊ နမူနာများ သို့မဟုတ် နှစ်ခုလုံးပေါင်းစပ်၍ သင့်လိုချင်သည်ကို ပေါ်လွင်စေပါ။ မော်ဒယ်အား ပုဒ်မတစ်စုကို အက္ခရာစဉ်အလိုက် စဉ်စားရန်၊ သို့မဟုတ် စိတ်ခံစားချက်အလိုက် အပိုဒ်ခွဲရန် ပြုလုပ်လိုပါက ၎င်းသည် သင်လိုချင်သည့် အကြောင်းဖြစ်ကြောင်း ပြသပေးပါ။

**အရည်အသွေးကောင်းမွန်သော ဒေတာများ ပေးပါ** ။ ကလက်စီဖိုင်ယာ တည်ဆောက်ရန် သို့မဟုတ် မော်ဒယ်အတိုင်းအတာ လိုက်နာရန် ကြိုးစားပါက နမူနာများ လုံလောက်စွာ ရှိထားပါစေ။ သင့်နမူနာများကို စစ်ဆေးခြင်းလုပ်ပါ- မော်ဒယ်သည် ပုံမှန်စာလုံးပေါင်းအမှားများကို ရှင်းလင်းစွာကြည့်နိုင်ပြီး တုံ့ပြန်ချက်ပေးနိုင်ပေမယ့်၊ ၎င်းကို ရည်ရွယ်ချက်ဖြစ်သလို ထင်မြင်ရစေသဖြင့် တုံ့ပြန်ချက်ထိခိုက်နိုင်သည်။

**သင့်ဆက်တင်များကိုစစ်ဆေးပါ** ။ temperature နှင့် top_p ဆက်တင်များသည် မော်ဒယ်၏ တုံ့ပြန်ချက်ထုတ်ပေးမှု၏ သေချာမှုကို ထိန်းချုပ်သည်။ တုံ့ပြန်ချက်တွင် တစ်ခုတည်းသော မှန်ကန်သောဖြေချက်ရှိပါက ၎င်းတို့ကို နိမ့်ချချင်မည်။ ပိုမိုစုံလင်သော တုံ့ပြန်ချက်များ ရရှိစေရန်မို့လျှင် အရမ်းမြင့်ချင်နိုင်သည်။ တစ်ခုတည်းသော လူတွေ့မှားသည် ဆက်တင်များကို " ထက်မြတ်မှု" သို့မဟုတ် "ဖန်တီးမှု" ထိန်းချုပ်ချက်ဟု ဟယူကြသည့် ဖြစ်ပေါ်မှုဖြစ်သည်။


ရင်းမြစ်- https://learn.microsoft.com/azure/ai-foundry/openai/overview


ပုံသည် သင်၏ ပထမဆုံး စာသား အချက်ပြကို ဖန်တီးနေသည်!


### ၅။ တင်သွင်းပါ!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### တူညီသောခေါ်ဆိုချက်ကို ထပ်မံခေါ်ဆိုပါက ရလဒ်များကို မည်သို့နှိုင်းယှဉ်နိုင်သနည်း?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## စာသားအကျဥ်းချုပ်  
#### စိန်ခေါ်မှု  
စာသားပိုဒ်တစ်ပိုဒ်အဆုံးတွင် 'tl;dr:' ကိုထည့်သွင်းခြင်းဖြင့် စာသားကို အကျဉ်းချုပ်ပေးပါ။ မော်ဒယ်သည် အပိုဆောင်းညွှန်ကြားချက်မလိုဘဲ အလုပ်များစွာကို ဘယ်လိုဆောင်ရွက်ရမည့်ကို သိရှိနားလည်ကြောင်း သတိပြုပါ။ မော်ဒယ်၏အပြုအမူကို ပြောင်းလဲရန်နှင့် သင်ရသော အကျဉ်းချုပ်ကို ကိုယ်ပိုင်ပြင်ဆင်နိုင်ရန် tl;dr ထက်ပိုမိုဖော်ပြချက်ပြည့်ဝသည့် prompt များနှင့် စမ်းသပ်နိုင်သည်(3)။  

နောက်ဆုံးအလုပ်များမှာ စာသားကြီးမားသည့် ကော်ပိုရာတစ်ခုတွင် ကြိုတင်လေ့ကျင့်ပြီး ထိုနောက် ထူးခြားသည့် အလုပ်တစ်ခုတွင် နူးညံ့စွာလေ့ကျင့်ခြင်းမှတဆင့် NLP အလုပ်များနှင့် benchmark များတွင် သိသာထင်ရှားသော တိုးတက်မှုများပြသခဲ့သည်။ ပုံမှန်အားဖြင့် အလုပ်မပေါ်သြဇာရှိသည့် ဖွဲ့စည်းတည်ဆောက်မှုပုံစံဖြစ်ပေမယ့် ထိုနည်းလမ်းသည် သာမန်အားဖြင့် သန်းပေါင်းများစွာ သို့မဟုတ် သိန်းပေါင်းများစွာ သင်ခန်းစာထည့်သွင်းချက်များကို လိုအပ်သည်။ ယင်းကိုနှိုင်းယှဉ်သောအခါ လူများသည် သာမန်အားဖြင့် အတွေးအခေါ်အနည်းငယ် သို့မဟုတ် ရိုးရှင်းသော နည်းပြချက်များမှသာ အသစ်သော ဘာသာစကားအလုပ်တစ်ခုကို ဆောင်ရွက်နိုင်သည် - သို့သော် လက်ရှိ NLP စနစ်များအများစုသည် အစိတ်အပိုင်းဖြစ်စရာ ရန်ကုန်ခွင့်မပြုသေးပါ။ ဤနေရာတွင် ဘာသာစကားမော်ဒယ်များကို ပိုမိုကြီးမားအောင် တိုးမြှင့်ခြင်းက အလုပ်အမျိုးအစားမရွေး အနည်းငယ်သော နမူနာဖြင့်ဆောင်ရွက်နိုင်မှုကို အလွန်တိုးတက်စေပြီး တခါတလေ ယခင်နည်းလမ်းဖြစ်သည့် နူးညံ့သော fine-tuning နည်းလမ်းများနှင့်ယှဉ်လျှင် နိုင်ပိုင်မှုရှိနေသည်ကို ပြသသည်။  



အကျဉ်းချုပ်  


# အသုံးပြုမှုအမျိုးမျိုးအတွက် မျှဝေမှုများ  
1. စာသားကို အကျဥ်းချုပ် ပြုစုပါ  
2. စာသား စုဝေး ခြယ်ခြားပါ  
3. ထုတ်ကုန်နာမည်အသစ်များ ဖန်တီးပါ  
4. အင်ဗိုက်ဒ်များ  
5. စုဝေးခြယ်ခြားသူတစ်ယောက် ကို ကိုယ်ပိုင်လေ့ကျင့်ပါ  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## စာသား သတ်မှတ်ခြင်း  
#### စိန်ခေါ်မှု  
ဝိဇာနနည်းအချိန်တွင် ပေးအပ်ထားသော category များအတိုင်း အရာဝတ္ထုများကို သတ်မှတ်ပါ။ အောက်ပါ ဥပမာတွင် ကျွန်ုပ်တို့သည် စာသားသတ်မှတ်ရန် category များနှင့် စာသားနှစ်ခုလုံးကို prompt(*playground_reference) တွင် ပေးထားသည်။  

မှာယူသူ စုံစမ်းမေးမြန်းမှု - မင်္ဂလာပါ၊ ကျွန်ုပ်၏လပ်တော့ ကီးဘုတ်ပေါ်ရှိ key တစ်ခု လတ်တလော ပျက်သွားပြီး အစားထိုးရန် လိုအပ်ပါသည်။  

သတ်မှတ်ထားသော category:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## ထုတ်ကုန်အမည်အသစ်များတီထွင်ခြင်း
#### စိန်ခေါ်မှု
ဥပမာအသုံးအနှုန်းများမှ ထုတ်ကုန်အမည်များ ဖန်တီးပါ။ ဤမှာ ငါတို့ထုတ်ကုန်အမည်များဖန်တီးရန် စာရင်းတင်နေသော ထုတ်ကုန်အကြောင်းအရာကိုအထောက်အထားအဖြစ် ထည့်သွင်းထားသည်။ ငါတို့သည် လက်ခံလိုသောနမူနာပုံစံကို ပြသရန် ညီထွေမှုတစ်ခုလည်း ပေးထားသည်။ နောက်ထပ်ကောင်းမွန်ပြီး စိတ်ကူးထူးခြားမှုများရရှိရန်အတွက် အပူချိန်တန်ဖိုးကို မြင့်မားစေရန် လည်း သတ်မှတ်ထားသည်။

ထုတ်ကုန်ဖော်ပြချက်- အိမ်တွင်းနွားနို့မျှကိတ်ထုတ်စက်တစ်လုံး
စပါးစာလုံးများ- အမြန်၊ ကျန်းမာရေးကောင်း၊ သေးငယ်သည်။
ထုတ်ကုန်အမည်များ- HomeShaker, Fit Shaker, QuickShake, Shake Maker

ထုတ်ကုန်ဖော်ပြချက်- ခြေနင်းအရွယ်အစားတစ်ခုစီနှင့် ကိုက်ညီသော ဖိနပ်စုံ
စပါးစာလုံးများ- သင့်လျော်သော၊ ကိုက်ညီသော၊ omni-fit။


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## အတွဲချိတ်ခြင်းများ (Embeddings)
ဤအပိုင်းတွင် အတွဲချိတ်ချက်များကို ရယူနည်းနှင့် စကားလုံးများ၊ စာကြောင်းများ၊ အကြောင်းအရာများအကြား ဆင်တူမှုပေါ်တည်၍ ရှာဖွေသည့်နည်းကို ပြသပါမည်။ အောက်ပါ နိုက်ဘုတ်များကို ကိုင်တွယ်ရန်အတွက် `text-embedding-ada-002` ကို အခြေခံမော်ဒယ်အဖြစ် အသုံးပြုသည့် မော်ဒယ်တစ်ခုကို တပ်ဆင်ပြီး၊ .env ဖိုင်အတွင်း၌ သူ၏ တပ်ဆင်သည့်အမည်အား `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` အကြောင်းပြုချက်ဖြင့် သတ်မှတ်ထားရမည်ဖြစ်သည်။


### မော်ဒယ်အမျိုးအစား - embedding မော်ဒယ် ရွေးချယ်ခြင်း

**မော်ဒယ်အမျိုးအစား**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> စာသား-embedding  (embedding မော်ဒယ် မိသားစု)  
{capability} --> ada             (အခြား embedding မော်ဒယ်များအားလုံးကို 2024 ခုနှစ်တွင် ရပ်ဆိုင်းမည်)  
{input-type} --> မသတ်မှတ်ပါ (ရှာဖွေရေး မော်ဒယ်များအတွက်သာ သတ်မှတ်သည်)  
{identifier} --> 002             (ဗားရှင်း 002)  

model = 'text-embedding-ada-002'


  > [!NOTE] Codespaces တွင် သို့မဟုတ် Devcontainer အတွင်းတွင် ဤ notebook ကို အလုပ်လုပ်ပါက အောက်ပါအဆင့်မှာ မလိုအပ်ပါ


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## CNN နေ့စဉ်သတင်းအချက်အလက်စုမှ ဆောင်းပါးနှိုင်းပြိုင်ခြင်း
source: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# ကိုးကားစာရင်း  
- [Microsoft Foundry - Azure OpenAI Models](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# အကူအညီပိုမိုလိုသေးပါက  
[OpenAI ကုန်စည်ဖြန့်ချိရေး အဖွဲ့](AzureOpenAITeam@microsoft.com)


# ပါဝင်ဆောင်ရွက်သူများ
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
